In [1]:
# ── Cell 1: Install ──────────────────────────────────────────────────────────
!pip install -q catboost scikit-learn pandas numpy


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python3.14 -m pip install --upgrade pip


In [2]:
# ── Cell 2: Imports ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, classification_report

SEEDS    = [42, 7, 123]
N_SPLITS = 10
print('Libraries loaded.')

Libraries loaded.


In [3]:
# ── Cell 3: Load data ────────────────────────────────────────────────────────
TRAIN_DATA  = pd.read_csv('train-data.csv',  index_col='id')
TRAIN_LABEL = pd.read_csv('train-label.csv', index_col='id')
TEST_DATA   = pd.read_csv('test-data.csv',   index_col='id')

print(f'Train: {TRAIN_DATA.shape}, Test: {TEST_DATA.shape}')
print(TRAIN_LABEL['disorder'].value_counts().sort_index())

Train: (13249, 41), Test: (8834, 41)
disorder
0     389
1    2068
2    1090
3    3096
4      58
5    1700
6     813
7    2643
8      91
9    1301
Name: count, dtype: int64


In [4]:
# ── Cell 4: Preprocessing (a3 original — best so far) ────────────────────────
def preprocess(df):
    df = df.copy()
    drop_cols = [
        'first_name', 'last_name', 'insitute_name', 'institute_location',
        'test_1', 'test_2', 'test_3', 'test_4', 'test_5', 'treatment_consent'
    ]
    df = df.drop(columns=drop_cols)

    miss_cols = [
        'gender', 'maternal_defect', 'mother_age', 'father_age',
        'respiration', 'heart_rate', 'risk_level', 'place_birth',
        'folic_acid', 'maternal_illness', 'infertility_treatment',
        'problem_previous_pregnancies', 'abortion_cnt',
        'birth_defects', 'white_blood_cell_count', 'blood_test',
        'symptom_1', 'symptom_2', 'symptom_3', 'symptom_4', 'symptom_5'
    ]
    df['missing_count']      = df[miss_cols].isna().sum(axis=1)
    df['missing_parent_age'] = df['mother_age'].isna().astype(int) + df['father_age'].isna().astype(int)
    df['missing_symptoms']   = df[['symptom_1','symptom_2','symptom_3','symptom_4','symptom_5']].isna().sum(axis=1)
    df['missing_clinical']   = df[['respiration','heart_rate','risk_level','blood_test']].isna().sum(axis=1)

    for col in ['mother_age','father_age','maternal_defect','gender',
                'risk_level','heart_rate','respiration','abortion_cnt','white_blood_cell_count']:
        df[f'{col}_missing'] = df[col].isna().astype(int)

    binary_yn = [
        'mother_defect','father_defect','maternal_defect','paternal_defect',
        'alive','folic_acid','maternal_illness','infertility_treatment',
        'problem_previous_pregnancies',
        'symptom_1','symptom_2','symptom_3','symptom_4','symptom_5'
    ]
    for col in binary_yn:
        df[col] = df[col].map({'Y': 1, 'N': 0})

    df['respiration']   = df['respiration'].map({'A': 1, 'N': 0})
    df['heart_rate']    = df['heart_rate'].map({'A': 1, 'N': 0})
    df['risk_level']    = df['risk_level'].map({'H': 1, 'L': 0})
    df['place_birth']   = df['place_birth'].map({'I': 1, 'H': 0})
    df['birth_defects'] = df['birth_defects'].map({'S': 1, 'M': 2})
    df['gender']        = df['gender'].map({'M': 0, 'F': 1, 'A': 2})
    df['autopsy']       = df['autopsy'].map({'Y': 1, 'N': 0})
    for col in ['birth_asphyxia', 'radiation_exposure', 'substance_abuse']:
        df[col] = df[col].map({'Y': 1, 'N': 0, 'NR': 2})
    df['blood_test'] = df['blood_test'].map({'N': 0, 'I': 1, 'S': 2, 'A': 3})

    df['defect_sum']  = df[['mother_defect','father_defect','maternal_defect','paternal_defect']].sum(axis=1)
    df['symptom_sum'] = df[['symptom_1','symptom_2','symptom_3','symptom_4','symptom_5']].sum(axis=1)
    df['defect_x_symptom']     = df['defect_sum'] * df['symptom_sum']
    df['any_defect']           = (df['defect_sum'] > 0).astype(int)
    df['any_symptom']          = (df['symptom_sum'] > 0).astype(int)
    df['high_symptom']         = (df['symptom_sum'] >= 4).astype(int)
    df['all_defects']          = (df['defect_sum'] == 4).astype(int)
    df['parent_age_gap']       = (df['father_age'] - df['mother_age']).abs()
    df['symptom_defect_ratio'] = df['symptom_sum'] / (df['defect_sum'] + 1)
    df['s4_and_s5']    = ((df['symptom_4'] == 1) & (df['symptom_5'] == 1)).astype(int)
    df['no_s4_s5']     = ((df['symptom_4'] == 0) & (df['symptom_5'] == 0)).astype(int)
    df['late_vs_early']= (df['symptom_4'].fillna(0) + df['symptom_5'].fillna(0)
                         - df['symptom_1'].fillna(0) - df['symptom_2'].fillna(0))
    df['weighted_sym'] = (df['symptom_1'].fillna(0)*1 + df['symptom_2'].fillna(0)*1 +
                          df['symptom_3'].fillna(0)*1 + df['symptom_4'].fillna(0)*2 +
                          df['symptom_5'].fillna(0)*2)
    df['both_parents_defect'] = ((df['mother_defect'] == 1) & (df['father_defect'] == 1)).astype(int)
    df['no_parent_defect']    = ((df['mother_defect'] == 0) & (df['father_defect'] == 0)).astype(int)
    return df

X_train = preprocess(TRAIN_DATA)
X_test  = preprocess(TEST_DATA)
y_train = TRAIN_LABEL['disorder'].values
print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')

X_train: (13249, 59), X_test: (8834, 59)


In [5]:
# ── Cell 5: Class weights ────────────────────────────────────────────────────
class_counts  = np.bincount(y_train)
class_weights = len(y_train) / (10 * class_counts)
print('Class weights:')
for i, (n, w) in enumerate(zip(class_counts, class_weights)):
    print(f'  Class {i}: weight={w:.3f}  (n={n})')

Class weights:
  Class 0: weight=3.406  (n=389)
  Class 1: weight=0.641  (n=2068)
  Class 2: weight=1.216  (n=1090)
  Class 3: weight=0.428  (n=3096)
  Class 4: weight=22.843  (n=58)
  Class 5: weight=0.779  (n=1700)
  Class 6: weight=1.630  (n=813)
  Class 7: weight=0.501  (n=2643)
  Class 8: weight=14.559  (n=91)
  Class 9: weight=1.018  (n=1301)


In [6]:
# ── Cell 6: Train base model — get OOF + test probabilities ──────────────────
# Same as the XGB blend run (CB=1.0) which gave LB 0.37127
# We need the test probabilities to generate pseudo-labels

print('Training base model (a3 pipeline)...')

base_oof_proba  = np.zeros((len(y_train), 10))
base_test_preds = np.zeros((len(X_test), 10))

for SEED in SEEDS:
    print(f"\n{'='*40} SEED={SEED} {'='*40}")
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    oof_proba   = np.zeros((len(y_train), 10))
    test_preds  = np.zeros((len(X_test), 10))
    fold_scores = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train[tr_idx],      y_train[val_idx]

        model = CatBoostClassifier(
            iterations=2000, learning_rate=0.03, depth=6, l2_leaf_reg=3,
            class_weights=class_weights, early_stopping_rounds=100,
            eval_metric='Accuracy', random_seed=SEED, verbose=0, thread_count=-1,
        )
        model.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)

        val_proba = model.predict_proba(X_val)
        score     = balanced_accuracy_score(y_val, np.argmax(val_proba, axis=1))
        fold_scores.append(score)
        print(f'  Fold {fold+1:2d}: BA={score:.4f}  best_iter={model.best_iteration_}')
        oof_proba[val_idx] += val_proba
        test_preds         += model.predict_proba(X_test) / N_SPLITS

    oof_score = balanced_accuracy_score(y_train, np.argmax(oof_proba, axis=1))
    print(f'  OOF BA (seed={SEED}): {oof_score:.4f} | mean={np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}')
    base_oof_proba  += oof_proba  / len(SEEDS)
    base_test_preds += test_preds / len(SEEDS)

base_oof_score = balanced_accuracy_score(y_train, np.argmax(base_oof_proba, axis=1))
print(f'\nBASE OOF BA: {base_oof_score:.4f}')

Training base model (a3 pipeline)...

======================================== SEED=42 ========================================
  Fold  1: BA=0.3791  best_iter=146
  Fold  2: BA=0.4142  best_iter=31
  Fold  3: BA=0.3981  best_iter=26
  Fold  4: BA=0.3728  best_iter=282
  Fold  5: BA=0.3991  best_iter=2
  Fold  6: BA=0.4520  best_iter=137
  Fold  7: BA=0.3511  best_iter=30
  Fold  8: BA=0.3710  best_iter=45
  Fold  9: BA=0.4241  best_iter=55
  Fold 10: BA=0.4243  best_iter=51
  OOF BA (seed=42): 0.3984 | mean=0.3986 ± 0.0291

======================================== SEED=7 ========================================
  Fold  1: BA=0.3853  best_iter=175
  Fold  2: BA=0.4051  best_iter=117
  Fold  3: BA=0.4169  best_iter=168
  Fold  4: BA=0.4145  best_iter=123
  Fold  5: BA=0.3705  best_iter=18
  Fold  6: BA=0.4045  best_iter=81
  Fold  7: BA=0.3728  best_iter=42
  Fold  8: BA=0.3782  best_iter=27
  Fold  9: BA=0.4131  best_iter=186
  Fold 10: BA=0.3779  best_iter=78
  OOF BA (seed=7): 0.3942

In [7]:
# ── Cell 7: Build pseudo-labeled dataset ─────────────────────────────────────
# Keep only test rows where the model is very confident (max_proba > threshold)
# These become additional labeled training rows
# We try two thresholds and pick the one with better OOF

PSEUDO_THRESHOLD = 0.85   # only add test rows where model is ≥85% confident

max_proba    = base_test_preds.max(axis=1)
pseudo_labels= np.argmax(base_test_preds, axis=1)

mask = max_proba >= PSEUDO_THRESHOLD
print(f'Threshold {PSEUDO_THRESHOLD}: {mask.sum()} test rows selected out of {len(mask)}')
print('Pseudo-label distribution:')
vals, cnts = np.unique(pseudo_labels[mask], return_counts=True)
for v, c in zip(vals, cnts):
    print(f'  Class {v}: {c} rows  (max_proba avg: {max_proba[mask][pseudo_labels[mask]==v].mean():.3f})')

# Build augmented training set
X_pseudo = X_test.iloc[mask].copy()
y_pseudo = pseudo_labels[mask]

X_train_aug = pd.concat([X_train, X_pseudo], axis=0).reset_index(drop=True)
y_train_aug = np.concatenate([y_train, y_pseudo])

print(f'\nOriginal train: {len(X_train)} rows')
print(f'Pseudo-labeled: {mask.sum()} rows')
print(f'Augmented train: {len(X_train_aug)} rows')

Threshold 0.85: 0 test rows selected out of 8834
Pseudo-label distribution:

Original train: 13249 rows
Pseudo-labeled: 0 rows
Augmented train: 13249 rows


In [8]:
# ── Cell 8: Experiment A — Pseudo-labeling only ───────────────────────────────
print('EXPERIMENT A: Pseudo-labeling only')
print('=' * 50)

# Recompute class weights on augmented dataset
aug_class_counts  = np.bincount(y_train_aug, minlength=10)
aug_class_weights = len(y_train_aug) / (10 * aug_class_counts)

pl_oof_proba  = np.zeros((len(y_train), 10))  # OOF only on original train rows
pl_test_preds = np.zeros((len(X_test), 10))

for SEED in SEEDS:
    print(f"\n{'='*40} SEED={SEED} {'='*40}")
    # Stratify split on original indices only (we validate on original train only)
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    oof_proba   = np.zeros((len(y_train), 10))
    test_preds  = np.zeros((len(X_test), 10))
    fold_scores = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
        # Training: original fold rows + ALL pseudo-labeled rows
        X_tr_orig = X_train.iloc[tr_idx]
        y_tr_orig = y_train[tr_idx]
        X_tr = pd.concat([X_tr_orig, X_pseudo], axis=0).reset_index(drop=True)
        y_tr = np.concatenate([y_tr_orig, y_pseudo])

        # Validation: original fold rows only (no pseudo-labels in val)
        X_val = X_train.iloc[val_idx]
        y_val = y_train[val_idx]

        model = CatBoostClassifier(
            iterations=2000, learning_rate=0.03, depth=6, l2_leaf_reg=3,
            class_weights=aug_class_weights, early_stopping_rounds=100,
            eval_metric='Accuracy', random_seed=SEED, verbose=0, thread_count=-1,
        )
        model.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)

        val_proba = model.predict_proba(X_val)
        score     = balanced_accuracy_score(y_val, np.argmax(val_proba, axis=1))
        fold_scores.append(score)
        print(f'  Fold {fold+1:2d}: BA={score:.4f}  best_iter={model.best_iteration_}')
        oof_proba[val_idx] += val_proba
        test_preds         += model.predict_proba(X_test) / N_SPLITS

    oof_score = balanced_accuracy_score(y_train, np.argmax(oof_proba, axis=1))
    print(f'  OOF BA (seed={SEED}): {oof_score:.4f} | mean={np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}')
    pl_oof_proba  += oof_proba  / len(SEEDS)
    pl_test_preds += test_preds / len(SEEDS)

pl_oof_score = balanced_accuracy_score(y_train, np.argmax(pl_oof_proba, axis=1))
print(f'\nEXP A (pseudo-label) OOF BA: {pl_oof_score:.4f}')
print(f'Base OOF BA:                  {base_oof_score:.4f}')
print(f'Change:                       {pl_oof_score - base_oof_score:+.4f}')

EXPERIMENT A: Pseudo-labeling only

======================================== SEED=42 ========================================
  Fold  1: BA=0.3791  best_iter=146
  Fold  2: BA=0.4142  best_iter=31
  Fold  3: BA=0.3981  best_iter=26
  Fold  4: BA=0.3728  best_iter=282
  Fold  5: BA=0.3991  best_iter=2
  Fold  6: BA=0.4520  best_iter=137
  Fold  7: BA=0.3511  best_iter=30
  Fold  8: BA=0.3710  best_iter=45
  Fold  9: BA=0.4241  best_iter=55
  Fold 10: BA=0.4243  best_iter=51
  OOF BA (seed=42): 0.3984 | mean=0.3986 ± 0.0291

======================================== SEED=7 ========================================
  Fold  1: BA=0.3853  best_iter=175
  Fold  2: BA=0.4051  best_iter=117
  Fold  3: BA=0.4169  best_iter=168
  Fold  4: BA=0.4145  best_iter=123
  Fold  5: BA=0.3705  best_iter=18
  Fold  6: BA=0.4045  best_iter=81
  Fold  7: BA=0.3728  best_iter=42
  Fold  8: BA=0.3782  best_iter=27
  Fold  9: BA=0.4131  best_iter=186
  Fold 10: BA=0.3779  best_iter=78
  OOF BA (seed=7): 0.3942 |

In [9]:
# ── Cell 9: Experiment B — Undersampling dominant classes ─────────────────────
# Undersample classes 1, 3, 7 (most dominant) down to ~600 each
# This reduces their dominance without throwing away all data
# Keep all minority classes untouched

print('EXPERIMENT B: Undersample dominant classes (1, 3, 7) to max 600')
print('=' * 60)

UNDERSAMPLE_CAP = 600   # max rows per dominant class
DOMINANT_CLASSES = [1, 3, 7]  # the three biggest classes

rng = np.random.default_rng(42)
keep_idx = []
for cls in range(10):
    cls_idx = np.where(y_train == cls)[0]
    if cls in DOMINANT_CLASSES and len(cls_idx) > UNDERSAMPLE_CAP:
        sampled = rng.choice(cls_idx, size=UNDERSAMPLE_CAP, replace=False)
        keep_idx.append(sampled)
        print(f'  Class {cls}: {len(cls_idx)} → {UNDERSAMPLE_CAP} (undersampled)')
    else:
        keep_idx.append(cls_idx)
        print(f'  Class {cls}: {len(cls_idx)} (kept all)')

keep_idx = np.concatenate(keep_idx)
X_train_us = X_train.iloc[keep_idx].reset_index(drop=True)
y_train_us = y_train[keep_idx]
print(f'\nOriginal: {len(X_train)} rows → Undersampled: {len(X_train_us)} rows')

us_class_counts  = np.bincount(y_train_us, minlength=10)
us_class_weights = len(y_train_us) / (10 * us_class_counts)

us_oof_proba  = np.zeros((len(y_train), 10))
us_test_preds = np.zeros((len(X_test), 10))

for SEED in SEEDS:
    print(f"\n{'='*40} SEED={SEED} {'='*40}")
    # Note: we split the undersampled set but evaluate OOF on ORIGINAL train
    # to compare fairly with other runs
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    oof_proba   = np.zeros((len(y_train), 10))
    test_preds  = np.zeros((len(X_test), 10))
    fold_scores = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
        # Map original val_idx into undersampled space — keep only those in keep_idx
        val_in_us   = np.intersect1d(val_idx, keep_idx)
        tr_in_us    = np.setdiff1d(keep_idx, val_in_us)

        # Remap to positional indices in X_train_us
        keep_set    = set(keep_idx)
        keep_list   = list(keep_idx)
        orig_to_us  = {orig: us for us, orig in enumerate(keep_list)}

        tr_us_idx   = [orig_to_us[i] for i in tr_in_us]
        val_us_idx  = [orig_to_us[i] for i in val_in_us]

        X_tr  = X_train_us.iloc[tr_us_idx]
        y_tr  = y_train_us[tr_us_idx]
        X_val = X_train_us.iloc[val_us_idx]
        y_val = y_train_us[val_us_idx]

        model = CatBoostClassifier(
            iterations=2000, learning_rate=0.03, depth=6, l2_leaf_reg=3,
            class_weights=us_class_weights, early_stopping_rounds=100,
            eval_metric='Accuracy', random_seed=SEED, verbose=0, thread_count=-1,
        )
        model.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)

        val_proba = model.predict_proba(X_val)
        score     = balanced_accuracy_score(y_val, np.argmax(val_proba, axis=1))
        fold_scores.append(score)
        print(f'  Fold {fold+1:2d}: BA={score:.4f}  best_iter={model.best_iteration_}')

        # Store OOF for original val_idx positions
        oof_proba[val_in_us] += model.predict_proba(X_train.iloc[val_in_us])
        test_preds            += model.predict_proba(X_test) / N_SPLITS

    # OOF only on rows that were in keep_idx (undersampled rows)
    oof_score = balanced_accuracy_score(y_train[keep_idx],
                                        np.argmax(oof_proba[keep_idx], axis=1))
    print(f'  OOF BA (seed={SEED}): {oof_score:.4f} | mean={np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}')
    us_oof_proba  += oof_proba  / len(SEEDS)
    us_test_preds += test_preds / len(SEEDS)

us_oof_score = balanced_accuracy_score(y_train[keep_idx],
                                       np.argmax(us_oof_proba[keep_idx], axis=1))
print(f'\nEXP B (undersample) OOF BA: {us_oof_score:.4f}')
print(f'Base OOF BA:                {base_oof_score:.4f}')

EXPERIMENT B: Undersample dominant classes (1, 3, 7) to max 600
  Class 0: 389 (kept all)
  Class 1: 2068 → 600 (undersampled)
  Class 2: 1090 (kept all)
  Class 3: 3096 → 600 (undersampled)
  Class 4: 58 (kept all)
  Class 5: 1700 (kept all)
  Class 6: 813 (kept all)
  Class 7: 2643 → 600 (undersampled)
  Class 8: 91 (kept all)
  Class 9: 1301 (kept all)

Original: 13249 rows → Undersampled: 7242 rows

======================================== SEED=42 ========================================
  Fold  1: BA=0.3902  best_iter=51
  Fold  2: BA=0.4214  best_iter=50
  Fold  3: BA=0.4346  best_iter=117
  Fold  4: BA=0.3767  best_iter=137
  Fold  5: BA=0.4009  best_iter=87
  Fold  6: BA=0.4377  best_iter=181
  Fold  7: BA=0.3636  best_iter=4
  Fold  8: BA=0.3642  best_iter=38
  Fold  9: BA=0.4175  best_iter=53
  Fold 10: BA=0.4126  best_iter=35
  OOF BA (seed=42): 0.4015 | mean=0.4020 ± 0.0260

======================================== SEED=7 ========================================
  Fold  1: 

In [10]:
# ── Cell 10: Per-class recall comparison ─────────────────────────────────────
disorder_names = {
    0:'레베르시', 1:'낭포성섬유증', 2:'당뇨', 3:'리증후군', 4:'암',
    5:'테이-삭스', 6:'혈색소침착증', 7:'사립체근병종', 8:'알츠하이머', 9:'확인안됨'
}
base_recall = {0:0.314, 1:0.392, 2:0.272, 3:0.351, 4:0.759,
               5:0.338, 6:0.488, 7:0.237, 8:0.571, 9:0.138}

def print_recall(proba, y_true, label):
    preds  = np.argmax(proba, axis=1)
    report = classification_report(y_true, preds, output_dict=True)
    ba     = balanced_accuracy_score(y_true, preds)
    print(f'\n{label}  (OOF BA={ba:.4f})')
    print(f'{"Class":<5} {"Name":<16} {"Base":>8} {"Now":>8} {"Δ":>7}')
    print('-' * 50)
    for cls in range(10):
        r     = report[str(cls)]['recall']
        r_old = base_recall[cls]
        delta = r - r_old
        flag  = ' ← up' if delta > 0.02 else ' ← LOW' if r < 0.3 else ''
        print(f'{cls:<5} {disorder_names[cls]:<16} {r_old:>8.3f} {r:>8.3f} {delta:>+7.3f}{flag}')

print_recall(base_oof_proba, y_train, 'BASE (no augmentation)')
print_recall(pl_oof_proba,   y_train, 'EXP A (pseudo-labeling)')
print_recall(us_oof_proba[keep_idx], y_train[keep_idx], 'EXP B (undersampling, on kept rows only)')


BASE (no augmentation)  (OOF BA=0.3860)
Class Name                 Base      Now       Δ
--------------------------------------------------
0     레베르시                0.314    0.314  -0.000
1     낭포성섬유증              0.392    0.392  +0.000
2     당뇨                  0.272    0.272  +0.000 ← LOW
3     리증후군                0.351    0.351  +0.000
4     암                   0.759    0.759  -0.000
5     테이-삭스               0.338    0.338  -0.000
6     혈색소침착증              0.488    0.488  +0.000
7     사립체근병종              0.237    0.237  -0.000 ← LOW
8     알츠하이머               0.571    0.571  +0.000
9     확인안됨                0.138    0.138  -0.000 ← LOW

EXP A (pseudo-labeling)  (OOF BA=0.3860)
Class Name                 Base      Now       Δ
--------------------------------------------------
0     레베르시                0.314    0.314  -0.000
1     낭포성섬유증              0.392    0.392  +0.000
2     당뇨                  0.272    0.272  +0.000 ← LOW
3     리증후군                0.351    0.351  +0.000
4     암

In [11]:
# ── Cell 11: Save best submission ────────────────────────────────────────────
# Compare and pick the better experiment
print(f'Base OOF:         {base_oof_score:.4f}  (LB known: 0.37127)')
print(f'Exp A pseudo-lab: {pl_oof_score:.4f}')
print(f'Exp B undersamp:  {us_oof_score:.4f}  (note: evaluated on fewer rows)')

# Save pseudo-labeled submission
pl_preds = np.argmax(pl_test_preds, axis=1)
sub_pl   = pd.DataFrame({'id': TEST_DATA.index, 'disorder': pl_preds}).set_index('id')
sub_pl.to_csv('submission_pseudolabel.csv')
print('\nSaved: submission_pseudolabel.csv')
print(sub_pl['disorder'].value_counts().sort_index())

# Save undersampled submission
us_preds = np.argmax(us_test_preds, axis=1)
sub_us   = pd.DataFrame({'id': TEST_DATA.index, 'disorder': us_preds}).set_index('id')
sub_us.to_csv('submission_undersample.csv')
print('\nSaved: submission_undersample.csv')
print(sub_us['disorder'].value_counts().sort_index())

print(f'\nSubmit pseudo-label if OOF > {base_oof_score:.4f}')
print(f'Submit undersample if OOF > {base_oof_score:.4f} (use caution — different eval set)')

Base OOF:         0.3860  (LB known: 0.37127)
Exp A pseudo-lab: 0.3860
Exp B undersamp:  0.3860  (note: evaluated on fewer rows)

Saved: submission_pseudolabel.csv
disorder
0     382
1    1392
2     633
3    1562
4     248
5    1287
6    1081
7    1168
8     242
9     839
Name: count, dtype: int64

Saved: submission_undersample.csv
disorder
0     355
1    1225
2     723
3    1725
4     283
5    1276
6    1073
7    1104
8     280
9     790
Name: count, dtype: int64

Submit pseudo-label if OOF > 0.3860
Submit undersample if OOF > 0.3860 (use caution — different eval set)
